In [ ]:
%load_ext sql
%sql sqlite:///Database_AAPL_2016_TO_2021.db

In [ ]:
%%sql

/*
# Objectif du code : s'entraîner aux sous-requêtes.
# Ici, on souhaite récupérer toutes les fois où le volume échangé sur la journée était supérieur au volume moyen sur la période entière.
*/
SELECT [date], [volume] FROM aapl_prices
WHERE [volume] >= (SELECT AVG([volume]) AS avg_volume FROM aapl_prices);





/*
# Problématique : qu'en est-il si l'on souhaite avoir les volumes qui excèdent les volumes moyens annuels ?
# c-à-d ne récupérer que les volumes qui dépassent le volume moyen de leur année respective.
# Réponse : on peut l'approcher de plusieurs manières, comme le montrent les deux codes qui suivent.
*/

/*
#1ere approche : tout écrire à la main, car on n'a pas de boucle 'for'
#Etapes : 4 codes

/*
# 1er code : création de la colonne "Annee" qui va recenser les années des observations. On le fait, car on va l'utiliser par la suite pour 
# calculer la moyenne des volumes sur chaque année.
*/
ALTER TABLE aapl_prices
ADD Annee INTEGER;



/*
# 2e code : on remplit la colonne "Annee".
*/
UPDATE aapl_prices
SET [Annee] = SUBSTRING([date], 1, 4);



/*
# 3e code : on associe, aux observations de chaque année, la moyenne des volumes sur leur année (grâce aux sous-requêtes).
# Le 3e code génère donc une table composée des colonnes suivantes : la date, l'année, le volume, le volume moyen selon l'année.

# Interprétation du code :
#     La commande CASE vient créer une nouvelle colonne, dans laquelle on va la remplir avec les volumes moyens annuels.
#     Concrètement, le CASE vient itérer sur chaque observation et lui associe une valeur (dans la nouvelle colonne), selon la condition dans le CASE qui est vérifiée.

#     Les lignes "WHEN [Annee] = x THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = x)" veulent dire :
#     pour cette observation, lorsque l'année est x ("WHEN [Annee] = x"), 
#     alors associez-lui le volume moyen de son année (sous-requête "SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = x").

#     La sous-requête "(SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = x)" renvoie le volume moyen de l'année x, 
#     donc une seule valeur et non une table, colonne ou autre (car on y a utilisé une fonction d'agrégation, et que les fonctions d'agrégation ne renvoient qu'UNE SEULE valeur).  
*/
SELECT [date], Annee, [volume],

CASE
    WHEN [Annee] = 2016 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2016)
    WHEN [Annee] = 2017 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2017)
    WHEN [Annee] = 2018 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2018)
    WHEN [Annee] = 2019 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2019)
    WHEN [Annee] = 2020 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2020)
    WHEN [Annee] = 2021 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2021)
END AS avg_volume

FROM aapl_prices;



/*
# 4e code : filtrage de la table créée par le 3e code, afin d'arriver au résultat voulu.
# Idée : le 3e code renvoie une table idéale et toute préparée à être filtrée. On l'utilise donc à la place de la table brute initiale aapl_prices.
# De ce fait, on se retrouve simplement à écrire une condition WHERE.
*/
SELECT [date], [Annee], [volume], [avg_volume] FROM 

    (SELECT [date], Annee, [volume],

    CASE
        WHEN [Annee] = 2016 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2016)
        WHEN [Annee] = 2017 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2017)
        WHEN [Annee] = 2018 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2018)
        WHEN [Annee] = 2019 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2019)
        WHEN [Annee] = 2020 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2020)
        WHEN [Annee] = 2021 THEN (SELECT AVG([volume]) AS avg_age FROM aapl_prices WHERE [Annee] = 2021)
    END AS avg_volume

    FROM aapl_prices)
    
WHERE [volume] >= [avg_volume];





/*
# 2e approche
# Le 4e code de la première approche nous donnait effectivement le résultat voulu, mais il y avait des lignes qu'on pouvait automatiser avec une boucle 'for'.
# Dans cette 2e approche, on se propose d'écrire cette boucle 'for', afin de gagner du temps.
# Pour cela, il faut déjà avoir une idée de la structure du code, ce qui peut être compliquée puisqu'on utilise une sous-requête assez "longue" à écrire à la main (donc à imaginer).
# Mais, on peut d'abord écrire une première partie à la main; et dès qu'on a une vision plus claire de ce qu'on doit écrire, on peut passer à la boucle 'for'.

# Ici, la ligne qu'on peut boucler est la ligne qui affiche le volume moyen annuel associé à l'année de l'observation : 
# "WHEN [Annee] = {i} THEN (SELECT AVG([volume]) AS avg_volume FROM aapl_prices WHERE [Annee] = {i}".

# On écrit le tout dans une variable "texte", qui correspond au script SQL qu'on aurait écrit à la main, mais ici écrit à l'aide de la boucle 'for'.
# On exécute ensuite ce script à l'aide de la fonction "read_sql" de pandas.
*/
import pandas as pd
import sqlite3

conn = sqlite3.connect("Database_AAPL_2016_TO_2021.db")

texte = "SELECT [date], [Annee], [volume], [avg_volume] FROM"
texte += "\n\n    (SELECT [date], [Annee], [volume],\n\n    CASE\n"

for i in range(2020, 2022+1,1):
    texte += f"        WHEN [Annee] = {i} THEN (SELECT AVG([volume]) AS avg_volume FROM aapl_prices WHERE [Annee] = {i})\n"

texte += "    END AS avg_volume\n\n"
texte += "    FROM aapl_prices)\n\n"
texte += "WHERE [volume] >= [avg_volume];"
    
print(texte)
pd.read_sql(texte, conn)





# 3e et 4e approches : avec une "sous-requête corrélée"
# Pour ces approches, on a besoin d'avoir une colonne 'Annee', qui va servir par la suite dans la sous-requête corrélée.

/*
# Création de la colonne 'Annee' et remplissage de la colonne avec les années des observations.
*/
ALTER TABLE aapl_prices
ADD Annee INTEGER;

UPDATE aapl_prices
SET Annee = SUBSTRING([date], 1, 4);


/*
# 3e approche
# Interprétation :
#     La sous-requête (SELECT AVG([volume]) FROM aapl_prices WHERE aapl_1.Annee = Annee) calcule le volume moyen (AVG([volume])) de chaque année (WHERE aapl_1.Annee = Annee).
#     En utilisant ces résultats dans une condition WHERE, on ne retient donc que les observations où le volume est >= au volume annuel moyen qui lui est associé.
*/
SELECT * FROM aapl_prices AS aapl_1
WHERE [volume] >= (SELECT AVG([volume]) FROM aapl_prices WHERE aapl_1.Annee = Annee);

/*
# 4e approche : plus de "vision/contrôle" que la 3e approche.
# Interprétation :
#     Création de la colonne 'avg_volume' qui va associer le volume annuel moyen propre à chaque observation, selon son année.
#     Cette association est réalisée grâce à la sous-requête corrélée : SET avg_volume_annee = (SELECT AVG([volume]) FROM aapl_prices WHERE aapl_1.Annee = Annee)
#     Après avoir remplit la colonne 'avg_volume', on n'a donc qu'à filtrer avec un simple WHERE

# La 4e approche apporte une vision plus précise de la base de données, puisqu'à chaque observation est associé le volume moyen annuel de son année d'observation.
# Cela nous permet donc de vérifier à l'oeil/à la main si les résultats sont cohérents, soit d'être plus confiants vis-à-vis des résultats. 
*/
ALTER TABLE aapl_prices
ADD avg_volume_annee FLOAT;

UPDATE aapl_prices AS aapl_1
SET avg_volume_annee = (SELECT AVG([volume]) FROM aapl_prices WHERE aapl_1.Annee = Annee);

SELECT * FROM aapl_prices
WHERE [volume] >= avg_volume_annee
*/

# Conclusion à la problématique :
# La problématique était de savoir comment on pouvait ne récupérer uniquement les observations pour lesquelles le volume était supérieur à la moyenne des volumes sur leur année.

# Première réponse (1ere et 2e approches) :
#   Intuitivement, je me suis dit qu'il fallait mélager une condition WHERE avec une boucle 'for'.
#   Je ne pensais pas que les sous-requêtes pouvait être utilisées ici, car je n'avais utilisé jusque-là que les sous-requêtes non-corrélées.
#   A cet égard, une première façon de répondre au problème consistait à écrire plein de lignes WHERE avec des sous-requêtes non-corrélées.
#   Cette méthode peut être réalisée à la main ou avec une boucle 'for'.

# Deuxième réponse (3e et 4e approches) :
#   Après avoir appris l'existence des sous-requêtes corrélées, et avoir vu des exemples similaires à la problématique ci-présente, j'ai testé les sous-requêtes corrélées.
#   Ou bien on répond directement à la problématique, sans vérification des résultats (3e approche).
#   Ou bien on répond à la problématique plus prudemment, avec des outils nous permettant de vérifier à la main la cohérence des résultats (4e approche).

#   Ainsi, la 4e approche est plus "sûre" que la 3e approche. Mais, elle nécessite de créer une colonne en plus, ce qui réprésente un certain coût.
#   Reste à voir si ce coût en vaut la peine...